

# 04 Regression

The following notebook builds on previous notebook. after having observed how different metrics differ given our ai classification strategy, we want to perform linear regression in order to see how ai as a classifier might impact our metrics in question 

----

growth dynamics

    - revenue growth 

    - employment growth

    - innovation, R&D 

financial health

    - profitability 

    - returns (on assets, investments)



In [5]:
# this file is used to find the project root and set the working directory to it.
from pathlib import Path
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt




In [6]:
def find_project_root(marker=".project-root"):
    path = Path.cwd().resolve()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker}")

PROJECT_ROOT = find_project_root()
print(f"Project root found at: {PROJECT_ROOT}")

cwd = PROJECT_ROOT



Project root found at: /Users/ducjeremyvu/mime/sem-1/files/bank-fintech/essay


In [7]:
# data.columns.to_list()

In [8]:
data = pd.read_pickle(cwd / "data/babina_cleaned_enriched.pkl")

selection = [

'gvkey',
'year',
'aiempl',
'totalempl',
'aiemp_gt_0',
'aiempl_ratio',
'ai_intensity',
'n_years',
'first_year',
'last_year',
'market_cap',
'revenue_growth',
'log_revenue_growth',
'employment_growth',
'roa',
'ni_over_totalempl',
'log_totalempl',
'log_aiemplratio',
'log_market_cap',
'log_roa'

]


data[selection].describe()

,gvkey,year,aiempl,totalempl,aiempl_ratio,n_years,first_year,last_year,market_cap,revenue_growth,log_revenue_growth,employment_growth,roa,ni_over_totalempl,log_totalempl,log_aiemplratio,log_market_cap,log_roa
count,50622.000000,50622.000000,50622.000000,5.062200e+04,50622.000000,50622.000000,50622.000000,50622.000000,50622.000000,42769.000000,42769.000000,42769.000000,50622.000000,50622.000000,50622.000000,16837.000000,50622.000000,44417.000000
mean,76430.701948,2011.316740,8.094386,6.576617e+03,0.000918,9.987673,2006.465272,2016.091363,28919.327781,4.496306,-0.001482,0.074747,0.208662,6.778142,6.812129,-6.949641,9.656472,-3.463792
std,73493.815943,3.975979,97.511928,2.717678e+04,0.004329,4.193476,2.723738,3.187312,25594.846322,113.021840,1.401740,0.185891,11.860670,38.261310,1.986406,1.465556,1.414921,1.411882
min,1004.000000,2005.000000,0.000000,1.000000e+00,0.000000,1.000000,2005.000000,2005.000000,0.417556,-0.999923,-9.471966,-0.752991,-19.073949,-142.479392,0.000000,-12.404415,-0.873335,-22.909164
25%,13839.000000,2008.000000,0.000000,2.360000e+02,0.000000,7.000000,2005.000000,2015.000000,7834.478710,-0.501700,-0.696554,0.009921,0.009154,0.085365,5.463832,-7.954021,8.966290,-4.157109
50%,31208.500000,2011.000000,0.000000,8.920000e+02,0.000000,11.000000,2005.000000,2018.000000,21567.542398,-0.002463,-0.002466,0.046440,0.026875,0.648391,6.793466,-6.944730,9.978945,-3.464727
75%,148224.000000,2015.000000,1.000000,3.227000e+03,0.000348,14.000000,2007.000000,2018.000000,43950.966878,0.981806,0.684009,0.097720,0.055233,3.233194,8.079308,-5.946075,10.690830,-2.762612
max,328795.000000,2018.000000,6190.000000,1.552543e+06,0.400000,14.000000,2018.000000,2018.000000,115691.966503,16051.910568,9.683645,11.600000,2456.927679,2122.653912,14.255405,-0.916291,11.658686,7.806667


In [9]:
import statsmodels.formula.api as smf

from linearmodels.panel import PanelOLS

from linearmodels.panel import compare


panel = data.set_index(["gvkey", "year"])

# Stargazer example

# Simple Regression

## Revenue Growth

### simple binary classifier

In [15]:
reg_data = data.dropna(subset=[
    # need to drop because during computation of growth rates, we get rows with NA for the first year of observation
    "revenue_growth",
    "aiemp_gt_0",
    "gvkey"
]).copy()

model_revgrowth_bin = smf.ols(
    "revenue_growth ~ aiemp_gt_0",
    data=reg_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gvkey"]}
)

print(model_revgrowth_bin.summary())

                            OLS Regression Results                            
Dep. Variable:         revenue_growth   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.3386
Date:                Fri, 26 Jun 2026   Prob (F-statistic):              0.561
Time:                        20:38:45   Log-Likelihood:            -2.6288e+05
No. Observations:               42769   AIC:                         5.258e+05
Df Residuals:                   42767   BIC:                         5.258e+05
Df Model:                           1                                         
Covariance Type:              cluster                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              4.2222      0

### Intensity Classifier

In [13]:
model_revgrowth_ai_int = smf.ols(
    "revenue_growth ~ ai_intensity",
    data=reg_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gvkey"]}
)

print(model_revgrowth_ai_int.summary())

                            OLS Regression Results                            
Dep. Variable:         revenue_growth   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                     1.283
Date:                Fri, 26 Jun 2026   Prob (F-statistic):              0.278
Time:                        20:37:19   Log-Likelihood:            -2.6288e+05
No. Observations:               42769   AIC:                         5.258e+05
Df Residuals:                   42765   BIC:                         5.258e+05
Df Model:                           3                                         
Covariance Type:              cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 17

### Ai Empl

In [14]:
model_revgrowth_ai_ratio = smf.ols(
    "revenue_growth ~ aiempl_ratio",
    data=reg_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gvkey"]}
)

print(model_revgrowth_ai_ratio.summary())

                            OLS Regression Results                            
Dep. Variable:         revenue_growth   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                  0.005938
Date:                Fri, 26 Jun 2026   Prob (F-statistic):              0.939
Time:                        20:37:45   Log-Likelihood:            -2.6288e+05
No. Observations:               42769   AIC:                         5.258e+05
Df Residuals:                   42767   BIC:                         5.258e+05
Df Model:                           1                                         
Covariance Type:              cluster                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        4.4905      0.550      8.160   

In [16]:
from statsmodels.iolib.summary2 import summary_col

summary_col(
    [model_revgrowth_bin, model_revgrowth_ai_int, model_revgrowth_ai_ratio],
    stars=True,
    model_names=["Simple Binary", "Intensity", "Ratio"],
    info_dict={
        "N": lambda x: f"{int(x.nobs)}",
        "R²": lambda x: f"{x.rsquared:.3f}"
    }
)

,Simple Binary,Intensity,Ratio
Intercept,4.2222***,17.4200,4.4905***
,(0.5356),(14.6703),(0.5503)
aiemp_gt_0[T.True],0.7452,,
,(1.2807),,
ai_intensity[T.Low AI],,-12.3618,
,,(14.7211),
ai_intensity[T.Mid AI],,-14.7571,
,,(14.6964),
ai_intensity[T.No AI],,-13.1978,
,,(14.6801),


# Adding for Controls